<a href="https://colab.research.google.com/github/eeeewyz/RAG/blob/main/6_chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ungraded Lab: Chunking
---
Welcome to the Ungraded Lab on Chunking! As you saw in the lectures, chunking breaks large texts into smaller, manageable pieces, which is essential for efficiently working with vector databases and language models.


# Table of Contents
- [ 1 - Introduction](#1)
  - [ 1.1 Importing necessary libraries](#1-1)
  - [ 1.2 Downloading the data](#1-2)
- [ 2 - Fixed-size chunking](#2)
  - [ 2.1 Example Chunking Code](#2-1)
  - [ 2.2 Chunking with overlap](#2-2)
- [ 3 - Variable-size chunking - Recursive Character Splitting](#3)
  - [ 3.1 Pseudo-code for variable-size chunking methods](#3-1)
  - [ 3.2 Mixing fixed and variable-sized chunking](#3-2)
- [ 4 - Chunking on real data](#4)
  - [ 4.1 Getting the data](#4-1)
  - [ 4.2 Chunking the chapters](#4-2)
  - [ 4.3 Loading Chunks into a Vector Database](#4-3)
- [ 5 - Searching ](#5)
- [ 6 - Incorporating in a RAG system](#6)


<a id='1'></a>
## 1 - Introduction

---

Chunking plays an important role in information retrieval. For example, when building a vector database from a collection of books, different chunk sizes can serve different purposes. Cataloging entire books as single vectors may help in identifying broad themes, but misses specific details. Chunking closer to the paragraph or sentence level enables the retrieval of specific information or concepts.

Language models typically have limitations on the amount of text they can process at once, known as the "context window." Chunking helps ensure that text inputs remain within these boundaries, allowing models to handle large documents, like novels, by splitting them into smaller sections.

In this ungraded lab you will explore ways of chunking and see how it can impact RAG systems!


<div align="center">
  <img src="images/chunking.png" alt="Overview" width="80%">
</div>

<a id='1-1'></a>
### 1.1 Importing necessary libraries


In [ ]:
from typing import List
import requests
import re
import weaviate
from weaviate.classes.config import Configure, Property, DataType, Tokenization
from weaviate.util import generate_uuid5
import tqdm
from weaviate.classes.query import Filter


In [ ]:
from utils import (
    generate_with_single_input,
    suppress_subprocess_output,
    kill_processes_on_ports
)

# Kill processes on ports before importing flask_app
# WARNING: Running this cell twice may kill the active kernel
kill_processes_on_ports([5000, 8080, 8097, 50050, 50051])

import flask_app

 * Serving Flask app 'flask_app'
 * Debug mode: off


<a id='1-2'></a>
### 1.2 Downloading the data

Now you need some text long enough to justify chunking. Let's take a part from the [Pro Git book](https://git-scm.com/book/en/v2) a specifically a chapter called "What is Git?"

In [ ]:
url = "https://raw.githubusercontent.com/progit/progit2/main/book/01-introduction/sections/what-is-git.asc"
source_text = requests.get(url).text

In [ ]:
print(source_text[:1000])

[[what_is_git_section]]
=== What is Git?

So, what is Git in a nutshell?
This is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you.
As you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid subtle confusion when using the tool.
Even though Git's user interface is fairly similar to these other VCSs, Git stores and thinks about information in a very different way, and understanding these differences will help you avoid becoming confused while using it.(((Subversion)))(((Perforce)))

==== Snapshots, Not Differences

The major difference between Git and any other VCS (Subversion and friends included) is the way Git thinks about its data.
Conceptually, most other systems store information as a list of file-based changes.
These other systems (CVS, Subversion, Perforce, and so o

In [ ]:
print(f"There are about {len(source_text.split())} words in this chapter. Depending on how your LLM tokenizes words, you'd expect roughly {round(len(source_text.split())*1.3)} tokens.")

There are about 1403 words in this chapter. Depending on how your LLM tokenizes words, you'd expect roughly 1824 tokens.


<a id='2'></a>
## 2 - Fixed-size chunking
---
Fixed-size chunking means breaking texts into pieces of the same size. For example, you might split an article into parts of 100 words each or sections of 200 characters each. This method is common because it is easy to use and works well.

It works by dividing texts into pieces that have a set number of units. These units can be words, characters, or even tokens. The number of units in each piece is the same up to a maximum limit, and there can be an optional overlap between the pieces.


<div align="center">
  <img src="images/fixed_size.png" alt="Fixed Size Chunking" width="80%">
</div>

<a id='2-1'></a>
### 2.1 Example Chunking Code

Let's see now an implementation for fixed-size chunking. There are many different implementations. The following implementation is a possible one.

整体流程

假设输入：

text = "I like machine learning and natural language processing very much"
chunk_size = 3

第一步，split()：

text_words = [
    "I",
    "like",
    "machine",
    "learning",
    "and",
    "natural",
    "language",
    "processing",
    "very",
    "much"
]

第二步，每三个单词取一次：

["I", "like", "machine"]

["learning", "and", "natural"]

["language", "processing", "very"]

["much"]

第三步，使用 " ".join() 重新连接：

[
    "I like machine",
    "learning and natural",
    "language processing very",
    "much"
]

In [ ]:
from typing import List

def get_chunks_fixed_size(text: str, chunk_size: int) -> List[str]:
    """
    按照固定的单词数量，将文本切分成多个 chunks。

    Args:
        text (str):
            需要切分的原始文本。

        chunk_size (int):
            每个 chunk 最多包含多少个单词。

    Returns:
        List[str]:
            切分后的文本块列表。
            每个元素都是一个字符串，最多包含 chunk_size 个单词。
    """

    # 使用空白字符将文本分割成一个个单词
    # 例如："I like machine learning"
    # 变成：["I", "like", "machine", "learning"]
    text_words = text.split()

    # 创建空列表，用于保存最终生成的所有 chunks
    chunks = []

    # 从索引 0 开始遍历单词列表
    # 每次向后移动 chunk_size 个位置
    #
    # 假设 len(text_words) = 10，chunk_size = 3
    # 那么 i 的取值为：0、3、6、9
    for i in range(0, len(text_words), chunk_size):

        # 从单词列表中取出当前 chunk 对应的单词
        # 切片范围是 i 到 i + chunk_size，但不包含右端点
        chunk_words = text_words[i:i + chunk_size]

        # 将单词列表重新连接成一个字符串
        # 单词之间使用空格连接
        chunk = " ".join(chunk_words)

        # 把当前生成的 chunk 添加到结果列表中
        chunks.append(chunk)

    # 返回所有切分完成的 chunks
    return chunks

In [ ]:
fixed_size_chunks = get_chunks_fixed_size(source_text, chunk_size = 100)

In [ ]:
print(len(fixed_size_chunks))

15


In [ ]:
fixed_size_chunks[0:3]

["[[what_is_git_section]] === What is Git? So, what is Git in a nutshell? This is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you. As you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid subtle confusion when using the tool. Even though Git's user interface is fairly similar to these other VCSs, Git stores and thinks about information in",
 'a very different way, and understanding these differences will help you avoid becoming confused while using it.(((Subversion)))(((Perforce))) ==== Snapshots, Not Differences The major difference between Git and any other VCS (Subversion and friends included) is the way Git thinks about its data. Conceptually, most other systems store information as a list of file-based changes. These other systems (CVS, Subversion, Perforce, and s

<a id='2-2'></a>
### 2.2 Chunking with overlap

Let's modify the code to allow overlapping, so chunks will have shared tokens.


<div align="center">
  <img src="images/overlap.png" alt="Chunking with overlap" width="80%">
</div>

简单例子

假设：

text = "A B C D E F G H I J K L"
chunk_size = 4
overlap_fraction = 0.5

重叠单词数量：

overlap_int = int(4 * 0.5)
            = 2

循环中，i 的取值为：

0, 4, 8
第一个 chunk
i = 0

切片范围：

text_words[max(0 - 2, 0) : 0 + 4]
text_words[0:4]

结果：

A B C D
第二个 chunk
i = 4

切片范围：

text_words[max(4 - 2, 0) : 4 + 4]
text_words[2:8]

结果：

C D E F G H

其中 C D 是与前一个 chunk 重复的内容。

第三个 chunk
i = 8

切片范围：

text_words[max(8 - 2, 0) : 8 + 4]
text_words[6:12]

结果：

G H I J K L

最终结果：

[
    "A B C D",
    "C D E F G H",
    "G H I J K L"
]

In [ ]:
from typing import List

def get_chunks_fixed_size_with_overlap(
    text: str,
    chunk_size: int,
    overlap_fraction: float
) -> List[str]:
    """
    按照固定的单词数量切分文本，并让相邻的 chunks 之间保留一定比例的重叠内容。

    Parameters:
        text (str):
            需要切分的原始文本。

        chunk_size (int):
            每次向后移动的单词数量，也是每个基础 chunk 的大小。

        overlap_fraction (float):
            相邻 chunks 的重叠比例。

            例如：
            chunk_size = 10
            overlap_fraction = 0.2

            则重叠单词数为：
            10 × 0.2 = 2 个单词。

    Returns:
        List[str]:
            切分完成的文本块列表。
            相邻的文本块之间会包含重复的单词。
    """

    # 使用空白字符切分文本，将字符串转换成单词列表
    #
    # 例如：
    # "I like machine learning"
    # 变成：
    # ["I", "like", "machine", "learning"]
    text_words = text.split()

    # 计算相邻 chunks 之间需要重复多少个单词
    #
    # 例如：
    # chunk_size = 10
    # overlap_fraction = 0.2
    # overlap_int = int(10 × 0.2) = 2
    #
    # int() 会舍弃小数部分
    # 例如 int(2.8) 的结果是 2
    overlap_int = int(chunk_size * overlap_fraction)

    # 创建空列表，用于保存最终生成的所有 chunks
    chunks = []

    # 从索引 0 开始遍历单词列表
    # 每次向后移动 chunk_size 个位置
    #
    # 例如：
    # chunk_size = 5
    # i 的取值可能是：0、5、10、15……
    for i in range(0, len(text_words), chunk_size):

        # 确定当前 chunk 的切片范围
        #
        # 正常情况下，当前基础范围是：
        # text_words[i : i + chunk_size]
        #
        # 为了和前一个 chunk 重叠，需要把起点向前移动 overlap_int：
        # i - overlap_int
        #
        # max(i - overlap_int, 0)：
        # 防止起点小于 0，尤其是第一个 chunk
        #
        # 切片的右端点 i + chunk_size 不包含在结果中
        chunk_words = text_words[
            max(i - overlap_int, 0) : i + chunk_size
        ]

        # 将当前单词列表重新连接成一个字符串
        # 单词之间使用空格连接
        chunk = " ".join(chunk_words)

        # 将当前生成的 chunk 添加到结果列表
        chunks.append(chunk)

    # 返回所有切分完成的 chunks
    return chunks

In [ ]:
for chosen_size in [5, 25, 100]:
    chunks = get_chunks_fixed_size_with_overlap(source_text, chosen_size, overlap_fraction=0.2)
    # Print outputs to screen
    print(f"\nSize {chosen_size} - {len(chunks)} chunks returned.")
    for i in range(3):
        print(f"Chunk {i+1}: {chunks[i]}")


Size 5 - 281 chunks returned.
Chunk 1: [[what_is_git_section]] === What is Git?
Chunk 2: Git? So, what is Git in
Chunk 3: in a nutshell? This is an

Size 25 - 57 chunks returned.
Chunk 1: [[what_is_git_section]] === What is Git? So, what is Git in a nutshell? This is an important section to absorb, because if you understand what Git
Chunk 2: if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you. As you learn Git, try to
Chunk 3: you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid

Size 100 - 15 chunks returned.
Chunk 1: [[what_is_git_section]] === What is Git? So, what is Git in a nutshell? This is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you. As you learn Git, try to clear your mind of

Note that the smaller chunks of text are very detailed, but they might **not have enough information to be useful for searching**. In contrast, **larger chunks start to contain more information, similar to a typical paragraph in length**. As these chunks become even longer, **their associated vector embeddings become more general**. Eventually, they reach a point where they are no longer effective for information searching.

<a id='3'></a>
## 3 - Variable-size chunking - Recursive Character Splitting

---
Now let's examine variable-size chunking. Unlike fixed-size chunking, the size of each chunk here is a result, not a starting point. In variable-size chunking, text is divided using a specific marker. This marker could be something like a sentence or paragraph break or even a structural element like a markdown header.

<div align="center">
  <img src="images/recursive.png" alt="Recursive Character Splitting" width="80%">
</div>

<a id='3-1'></a>
### 3.1 Pseudo-code for variable-size chunking methods

The simplest one is to split into paragraphs (`\n\n`)

In [ ]:
from typing import List


# 按“段落”切分文本
def get_chunks_by_paragraph(source_text: str) -> List[str]:
    """
    将文本按照段落切分成多个 chunks。

    参数：
        source_text: str
            输入的原始文本，类型是字符串。

    返回：
        List[str]
            返回一个字符串列表，每个元素是一个段落。

    注意：
        -> List[str] 是返回值类型提示，
        可以理解为一种“类型备注”，表示函数预计返回字符串列表。
        它不是强制执行的 Python 语法检查。
    """

    # split("\n\n") 表示遇到两个连续的换行符时进行切分
    # \n 表示换行
    # \n\n 通常表示中间有一个空行，也就是段落之间的分隔
    #
    # 例如：
    # "第一段\n\n第二段"
    #
    # 会得到：
    # ["第一段", "第二段"]
    return source_text.split("\n\n")

Another way, in this context, is to split into sections. As you can see inspecting the text, sections are divided with `\n==` markers.

In [ ]:
from typing import List


# 按 AsciiDoc 的章节标记切分文本
def get_chunks_by_asciidoc_sections(source_text: str) -> List[str]:
    """
    将 AsciiDoc 格式的文本按照 section 章节切分成多个 chunks。

    参数：
        source_text: str
            输入的 AsciiDoc 文本。

    返回：
        List[str]
            返回一个字符串列表，每个元素通常对应一个 section。

    注意：
        -> List[str] 是返回值类型提示，
        表示函数预计返回字符串列表。
    """

    # split("\n==") 表示遇到“换行符 + 两个等号”时进行切分
    #
    # 这里的 == 不是 Python 语法，
    # 而是 AsciiDoc 文档格式中的章节标题标记。
    #
    # AsciiDoc 中常见的标题格式：
    # = 主标题
    # == 二级章节
    # === 三级章节
    #
    # 所以，并不是所有文章的章节前面都一定有两个等号。
    # 只有使用 AsciiDoc 格式，并且 section 用 == 标记时，
    # 才适合使用 "\n==" 作为切分符。
    #
    # 例如：
    # "Introduction\n==Chapter 1\n正文\n==Chapter 2\n正文"
    #
    # 会得到类似：
    # [
    #     "Introduction",
    #     "Chapter 1\n正文",
    #     "Chapter 2\n正文"
    # ]
    #
    # 注意：split() 会删除作为分隔符的 "\n=="，
    # 因此切分结果中不会保留这两个等号。
    return source_text.split("\n==")

# repr() 简单例子与用法总结
# 1. 简单例子
text = "Hello\nWorld"

print(repr(text))

# 输出：
# 'Hello\nWorld'

# 区别：

# print(text)：按照正常格式显示，\n 会变成换行。
# repr(text)：显示字符串内部的真实结构，\n 会直接显示出来。

In [ ]:
for marker in ["\n\n", "\n=="]:
    chunks = source_text.split(marker)
    # Print outputs to screen
    print(f"\nUsing the marker: {repr(marker)} - {len(chunks)} chunks returned.")
    for i in range(3):
        print(f"Chunk {i+1}: {repr(chunks[i])}")


Using the marker: '\n\n' - 31 chunks returned.
Chunk 1: '[[what_is_git_section]]\n=== What is Git?'
Chunk 2: "So, what is Git in a nutshell?\nThis is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you.\nAs you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid subtle confusion when using the tool.\nEven though Git's user interface is fairly similar to these other VCSs, Git stores and thinks about information in a very different way, and understanding these differences will help you avoid becoming confused while using it.(((Subversion)))(((Perforce)))"
Chunk 3: '==== Snapshots, Not Differences'

Using the marker: '\n==' - 7 chunks returned.
Chunk 1: '[[what_is_git_section]]'
Chunk 2: "= What is Git?\n\nSo, what is Git in a nutshell?\nThis is an important section to absorb,

One noticeable issue with simple marker-based chunking is that **headings often become separate chunks**, which might not be ideal. In practice, you might use a mixed strategy by attaching short chunks, like headings, to the following chunk. This way, the heading stays connected to its relevant section. Let's explore this approach further.

<a id='3-2'></a>
### 3.2 Mixing fixed and variable-sized chunking

You can combine fixed-size and variable-size chunking to take advantage of both methods. For instance, use a variable-size chunker to divide text at paragraph markers, and then apply a fixed-size filter. If a chunk is too small, you can merge it with the next one, and if a chunk is too large, you can split it in the middle or at another marker within the chunk.

整体逻辑

这段代码不是单纯按照固定大小切分，而是：

先按 section 切分
        ↓
检查 section 是否太短
        ↓
太短 → 暂存在 buffer 中
        ↓
和下一个 section 合并
        ↓
达到最小长度后加入结果

例如原始章节切分后得到：

chunks = [
    "第一部分，10个单词",
    "第二部分，8个单词",
    "第三部分，20个单词"
]

假设：

min_length = 25

处理过程：

第一部分：10个单词
不足25 → 放入 chunk_buffer

第一部分 + 第二部分：18个单词
仍不足25 → 继续放入 chunk_buffer

第一部分 + 第二部分 + 第三部分：38个单词
达到25 → 加入 new_chunks

最终：

new_chunks = [
    "第一部分 + 第二部分 + 第三部分"
]
chunk_buffer 的作用

可以把它理解成一个临时存放区：

chunk 太短
   ↓
先放进 chunk_buffer
   ↓
等待和后面的 chunk 合并

In [ ]:
def mixed_chunking(source_text):
    """
    使用“章节切分 + 小块合并”的方式对文本进行 mixed chunking。

    处理流程：
    1. 先按照 AsciiDoc 的章节标记 "\n==" 切分文本；
    2. 检查每个 chunk 的单词数量；
    3. 如果 chunk 太短，就暂时保存到 chunk_buffer；
    4. 将它和下一个 chunk 合并；
    5. 当合并后的长度达到 min_length 后，再加入最终结果。

    Args:
        source_text (str):
            需要切分的原始文本。

    Returns:
        list:
            切分完成的文本块列表。
    """

    # 先按照 AsciiDoc 的 section 标记切分文本
    #
    # "\n==" 表示：
    # 一个换行符 + 两个等号
    #
    # 这里的 == 不是 Python 语法，
    # 而是 AsciiDoc 文档中的章节标题标记。
    chunks = source_text.split("\n==")

    # 用于保存最终处理完成的 chunks
    new_chunks = []

    # 临时缓冲区
    # 用来暂时保存长度太短、不能单独成为 chunk 的内容
    chunk_buffer = ""

    # 设置 chunk 的最小单词数量
    # 如果不足 25 个单词，就继续和下一个 chunk 合并
    min_length = 25

    # 依次处理前面按照章节切分出来的每个 chunk
    for chunk in chunks:

        # 将缓冲区中的内容和当前 chunk 合并
        #
        # 如果 chunk_buffer 是空字符串：
        # new_buffer 就基本等于当前 chunk
        #
        # 如果 chunk_buffer 中有之前留下的短 chunk：
        # 就会把之前的短 chunk 和当前 chunk 合并
        new_buffer = chunk_buffer + chunk

        # 按照空格将文本切分成单词列表
        #
        # 例如：
        # "I like machine learning"
        #
        # 变成：
        # ["I", "like", "machine", "learning"]
        new_buffer_words = new_buffer.split(" ")

        # 检查当前合并结果是否少于 25 个单词
        if len(new_buffer_words) < min_length:

            # 如果太短，就先不加入最终结果
            # 而是把它保存在 chunk_buffer 中
            #
            # 下一次循环时，再与下一个 chunk 合并
            chunk_buffer = new_buffer

        else:
            # 如果单词数量已经达到最小长度，
            # 就把当前内容加入最终 chunks
            new_chunks.append(new_buffer)

            # 当前内容已经保存，因此清空缓冲区
            chunk_buffer = ""

    # 循环结束后，检查缓冲区中是否还有剩余内容
    if len(chunk_buffer) > 0:

        # 即使最后剩余的内容不足 25 个单词，
        # 因为已经没有下一个 chunk 可以合并，
        # 所以仍然把它加入最终结果
        new_chunks.append(chunk_buffer)

    # 返回处理完成的 chunks
    return new_chunks

In [ ]:
mixed_chunks = mixed_chunking(source_text)
for i in range(3):
    print(f"Chunk {i+1}: {repr(mixed_chunks[i])}")

Chunk 1: "[[what_is_git_section]]= What is Git?\n\nSo, what is Git in a nutshell?\nThis is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you.\nAs you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid subtle confusion when using the tool.\nEven though Git's user interface is fairly similar to these other VCSs, Git stores and thinks about information in a very different way, and understanding these differences will help you avoid becoming confused while using it.(((Subversion)))(((Perforce)))\n"
Chunk 2: "== Snapshots, Not Differences\n\nThe major difference between Git and any other VCS (Subversion and friends included) is the way Git thinks about its data.\nConceptually, most other systems store information as a list of file-based changes.\nThese other systems (CVS, Subv

This strategy helps ensure that chunks are not too small while still using syntactic markers, like headings, to define boundaries. After examining chunking strategies on one text, let's explore how they perform on a larger collection of texts.

<a id='4'></a>
## 4 - Chunking on real data

---
In this and the following section, there will be comprehensive examples of chunking in practice. You will process several sections of the [Pro Git book](https://git-scm.com/book/en/v2) using different chunking methods and then compare how well each method performs in search tasks.


<a id='4-1'></a>
### 4.1 Getting the data

Let's get the entire 14 chapter book.

指定 GitHub API 地址
        ↓
依次访问两个章节的 sections 目录
        ↓
获取目录下的文件列表
        ↓
过滤掉文件夹，只保留文件
        ↓
根据 download_url 下载每个文件的正文
        ↓
提取章节名和文件名作为 metadata
        ↓
整理成字典并放入 list




最后返回的数据大概是：

[
    {
        "body": "文件1的完整正文……",
        "chapter_title": "01-introduction",
        "filename": "ch01-introduction.asc"
    },
    {
        "body": "文件2的完整正文……",
        "chapter_title": "01-introduction",
        "filename": "ch02-history.asc"
    }
]

Python 支持从列表末尾开始索引：

[-1]：最后一个元素
[-2]：倒数第二个元素
[-3]：倒数第三个元素

例如：

url = "book/01-introduction/sections/test.asc"

parts = url.split("/")

结果：

[
    "book",
    "01-introduction",
    "sections",
    "test.asc"
]

那么：

parts[-3]   # "01-introduction"
parts[-1]   # "test.asc"


response = requests.get("https://example.com")
html_text = response.text


requests.get() 返回的是一个 Response dui xiang

response.text

表示取出响应中的文本内容，返回类型是字符串 str。

可以简单记成：

response        → 整个网络响应对象
response.text   → 响应中的文本正文

In [ ]:
import requests


def get_book_text_objects():
    """
    从 GitHub 上的 Pro Git 仓库中读取指定章节的 section 文件，
    并将每个文件的正文和 metadata 整理成字典对象。

    Returns:
        list:
            返回一个字典列表。

            每个字典大致包含：
            {
                "body": 文件正文,
                "chapter_title": 章节名称,
                "filename": 文件名
            }
    """

    # 用于保存最终得到的所有文本对象
    text_objs = list()

    # GitHub API 的基础地址
    api_base_url = (
        "https://api.github.com/repos/progit/progit2/contents/book"
    )

    # 需要读取的章节路径
    # 每个路径对应 book 文件夹中的一个章节 sections 目录
    chapter_urls = [
        "/01-introduction/sections",
        "/02-git-basics/sections"
    ]

    # 依次处理每一个章节目录
    for chapter_url in chapter_urls:

        # 将基础 URL 和章节路径拼接起来
        #
        # 例如：
        # https://api.github.com/repos/progit/progit2/contents/book
        # +
        # /01-introduction/sections
        # 得到：
        # https://api.github.com/repos/progit/progit2/contents/book/01-introduction/sections
        #
        # requests.get() 向 GitHub API 发送 GET 请求
        response = requests.get(api_base_url + chapter_url)

        # response.json()：
        # 将 GitHub API 返回的 JSON 数据转换成 Python 对象
        # 这里通常会得到一个 list，
        # list 中的每个元素都是一个 dict，
        # 表示当前目录中的一个文件或子目录

        for file_info in response.json():

            #file_info 大概是一个 字典 dict，表示 GitHub API 返回的“某一个文件的信息”。
            # 例如：
            # file_info = {
            #     "name": "ch01-introduction.asc",
            #     "path": "book/01-introduction/sections/ch01-introduction.asc",
            #     "type": "file",
            #     "size": 5234,
            #     "download_url": "https://raw.githubusercontent.com/progit/progit2/main/book/01-introduction/sections/ch01-introduction.asc"
            # }

            if file_info["type"] == "file":

                # file_info["download_url"] 是文件原始内容的下载地址
                # 再次发送 GET 请求，获取文件正文
                file_response = requests.get(
                    file_info["download_url"]
                )

                # 假设 URL 类似：
                # .../book/01-introduction/sections/ch01-introduction.asc
                # split("/") 后大致得到：
                # [
                #   ...,
                #   "01-introduction",
                #   "sections",
                #   "ch01-introduction.asc"
                # ]
                #
                # [-3] 表示倒数第三个元素：
                # "01-introduction"

                chapter_title = (
                    file_info["download_url"]
                    .split("/")[-3]
                )

                filename = (
                    file_info["download_url"]
                    .split("/")[-1]
                )

                # 创建一个字典对象
                #
                # body：
                # 文件正文
                #
                # chapter_title：
                # 章节名称，属于 metadata
                #
                # filename：
                # 原文件名，属于 metadata
                text_obj = {
                    "body": file_response.text,
                    "chapter_title": chapter_title,
                    "filename": filename
                }

                # 将当前文件对象加入最终列表
                text_objs.append(text_obj)

    # 返回所有文件对应的文本对象
    return text_objs

In [ ]:
# This will generate a list with 14 elements, one for each chapter
book_text_objs = get_book_text_objects()

In [ ]:
print(book_text_objs[0].keys())

dict_keys(['body', 'chapter_title', 'filename'])


<a id='4-2'></a>
### 4.2 Chunking the chapters

The following chunking methods will be applied to each section:

- **Fixed-length chunks with 20% overlap:**
  - Chunks with 25 words each
  - Chunks with 100 words each

- **Variable-length chunks** using paragraph markers

- **Mixed-strategy chunks** using paragraph markers with a minimum chunk length of 25 words

Additionally, metadata will be added to each chunk, including the filename, chapter name, and chunk number.

它的核心作用是：

给切分后的每个 chunk 补充来源信息和顺序编号。

例如输入：

book_text_obj = {
    "chapter_title": "01-introduction",
    "filename": "about-version-control.asc"
}

chunks = [
    "Git is a version control system.",
    "It helps developers track changes."
]

输出大致为：

[
    {
        "chapter_title": "01-introduction",
        "filename": "about-version-control.asc",
        "chunk": "Git is a version control system.",
        "chunk_index": 0
    },
    {
        "chapter_title": "01-introduction",
        "filename": "about-version-control.asc",
        "chunk": "It helps developers track changes.",
        "chunk_index": 1
    }
]

In [ ]:
def build_chunk_objs(book_text_obj, chunks):
    """
    Constructs a list of chunk objects from a given book text object
    and its associated chunks.

    Args:
        book_text_obj (dict): A dictionary containing metadata for the book text,
                              including 'chapter_title' and 'filename'.
        chunks (list): A list of chunks that represent parts of the book text.

    Returns:
        list: A list of dictionaries, each representing a chunk object
              with 'chapter_title', 'filename', 'chunk', and 'chunk_index'.
    """
    chunk_objs = list()  # Initialize an empty list to store chunk objects

    # Iterate over the chunks with an index
    for i, c in enumerate(chunks):
        # Create a dictionary for each chunk with its associated data
        chunk_obj = {
            "chapter_title": book_text_obj["chapter_title"],  # Chapter title from the book text object
            "filename": book_text_obj["filename"],            # Filename from the book text object
            "chunk": c,                                       # The actual chunk of text
            "chunk_index": i                                  # The index of the chunk in the list
        }
        # Append the constructed chunk object to the list
        chunk_objs.append(chunk_obj)

    # Return the list of chunk objects
    return chunk_objs

这四种分别是：

fixed_size_25
固定长度切分。每个 chunk 大约包含 25 个词，相邻 chunk 之间保留 20% overlap。
fixed_size_100
也是固定长度切分，但每个 chunk 大约包含 100 个词，同样保留 20% overlap。
para_chunks
按自然段切分。一个 paragraph 通常作为一个 chunk，长度不固定。
para_chunks_min_25
混合切分策略。先按 paragraph 切分，如果某些段落太短，就把它们合并，使 chunk 尽量至少达到 25 个词。

In [ ]:
# 根据不同的 chunking strategy，生成多组 chunk 对象
# 最终结果会按照 strategy_name 分类保存
chunk_obj_sets = dict()


# 遍历所有原始文档对象
# 每个 book_text_obj 通常代表一本书中的一个文件或 section
for book_text_obj in book_text_objs:

    # 取出当前文档的正文
    text = book_text_obj["body"]

    # 对同一份正文使用多种 chunking strategy
    # 每次循环会得到：
    # strategy_name：策略名称
    # chunks：该策略切分后得到的文本块列表
    for strategy_name, chunks in [

        # 固定长度切分：
        # 每个 chunk 大约 25 个词
        # 相邻 chunk 之间保留 20% overlap
        [
            "fixed_size_25",
            get_chunks_fixed_size_with_overlap(text, 25, 0.2)
        ],

        # 固定长度切分：
        # 每个 chunk 大约 100 个词
        # 相邻 chunk 之间保留 20% overlap
        [
            "fixed_size_100",
            get_chunks_fixed_size_with_overlap(text, 100, 0.2)
        ],

        # 按自然段切分：
        # 通常一个 paragraph 作为一个 chunk
        [
            "para_chunks",
            get_chunks_by_paragraph(text)
        ],

        # 混合切分：
        # 先按 paragraph 切分
        # 如果 paragraph 太短，则和其他段落合并
        # 让 chunk 尽量达到最小长度 25
        [
            "para_chunks_min_25",
            mixed_chunking(text)
        ]
    ]:

        # 给每个普通 chunk 添加 metadata，例如：
        # chapter_title、filename、chunk_index
        chunk_objs = build_chunk_objs(book_text_obj, chunks)

        # 如果这是第一次遇到该策略，
        # 就先在 chunk_obj_sets 中创建一个空列表
        if strategy_name not in chunk_obj_sets.keys():
            chunk_obj_sets[strategy_name] = list()

        # 将当前文档产生的所有 chunk objects
        # 添加到对应策略的列表中
        chunk_obj_sets[strategy_name] += chunk_objs

In [ ]:
print(chunk_obj_sets.keys())

dict_keys(['fixed_size_25', 'fixed_size_100', 'para_chunks', 'para_chunks_min_25'])


In [ ]:
chunk_type = 'fixed_size_25' # Change it to check the different chunks!
chunk_obj_sets[chunk_type][0:2]

[{'chapter_title': '01-introduction',
  'filename': 'about-version-control.asc',
  'chunk': '=== About Version Control (((version control))) What is "`version control`", and why should you care? Version control is a system that records changes to a',
  'chunk_index': 0},
 {'chapter_title': '01-introduction',
  'filename': 'about-version-control.asc',
  'chunk': 'that records changes to a file or set of files over time so that you can recall specific versions later. For the examples in this book, you will use software',
  'chunk_index': 1}]

<a id='4-3'></a>
### 4.3 Loading Chunks into a Vector Database

In this section, you'll focus on loading chunks into a vector database. Below, you'll find an outline of how to create and load data into the vector database. However, in this lab, you will work with a pre-loaded collection to save time. If you haven't yet completed the ungraded lab on the Weaviate API, it's highly recommended you do so for a better understanding of the process!

这段代码的作用：启动/连接 Weaviate 数据库，并指定它用外部的 Transformer API 来生成向量。

具体步骤：
先清理端口
   ↓
设置数据库保存路径
   ↓
优先启动 embedded Weaviate
   ↓
如果失败，就连接本地 Weaviate 8079 / 50050
   ↓
如果还失败，就尝试 8080 / 50051

In [ ]:
import os

# 1. 先杀掉可能占用 Weaviate 端口的进程
# Weaviate 常用端口：
# 8080 / 8079: HTTP API 端口
# 50050 / 50051: gRPC 端口
kill_processes_on_ports([8080, 8079, 50050, 50051])


# 2. 设置 Weaviate 数据保存的位置
# 从环境变量 COLLECTION_M3 读取基础路径
# 如果没有这个环境变量，就默认用当前目录 './'
collection_base = os.getenv('COLLECTION_M3', './')

# 最终的数据持久化路径：
# 例如 ./ungraded_lab_2
persistence_path = os.path.join(collection_base, 'ungraded_lab_2')


# 3. 尝试连接 embedded Weaviate
# embedded Weaviate = 在当前 Python 程序里启动一个本地 Weaviate
with suppress_subprocess_output():
    try:
        client = weaviate.connect_to_embedded(
            persistence_data_path=persistence_path,  # 数据保存路径
            environment_variables={
                # 开启基于 API 的向量化模块
                "ENABLE_API_BASED_MODULES": "true",

                # 启用 text2vec-transformers 模块
                # 也就是用 Transformer 模型把文本转成向量
                "ENABLE_MODULES": 'text2vec-transformers',

                # 指定 Transformer 推理服务地址
                # Weaviate 会把文本发到这个 Flask/API 服务
                # 然后拿回 embedding 向量
                "TRANSFORMERS_INFERENCE_API": "http://127.0.0.1:5000/",
            }
        )

    except Exception as e:
        # 如果 embedded Weaviate 连接失败，打印错误
        print(f"Failed to connect to embedded Weaviate: {e}")
        print("Falling back to local Weaviate connection...")

        try:
            # 4. 尝试连接本地已经运行的 Weaviate
            # 使用 8079 和 50050 这组端口
            client = weaviate.connect_to_local(
                port=8079,
                grpc_port=50050
            )

        except Exception as e2:
            # 如果这组端口也失败，继续尝试另一组默认端口
            print(f"Failed to connect to local Weaviate: {e2}")
            print("Trying alternative ports...")

            # 5. 最后尝试连接 8080 和 50051
            client = weaviate.connect_to_local(
                port=8080,
                grpc_port=50051
            )

这段代码的作用是：创建一个名叫 chunking_example 的 Weaviate collection，用来存储不同 chunking 方法切出来的文本块，并为这些文本块生成向量。

In [ ]:
# Creating the collection
# 创建一个 collection，相当于创建一张“表”
# 这里的 collection 名字叫 chunking_example

if not client.collections.exists("chunking_example"):
    # 如果这个 collection 还不存在，就创建它
    collection = client.collections.create(
        name='chunking_example',

        # 配置向量化方式
        vectorizer_config=[
            Configure.NamedVectors.text2vec_transformers(
                name="vector",
                # name="vector" 表示给这个向量字段起名叫 vector
                # 之后查询或访问向量时，会用到这个名字

                # source_properties=['chunk'],
                # 这里被注释掉了
                # 如果打开，意思是只用 chunk 这个字段生成向量
                # 如果不写 source_properties，通常会用可向量化的文本字段一起生成向量

                vectorize_collection_name=False,
                # 是否把 collection 的名字也加入向量化文本
                # False 表示不要把 "chunking_example" 这个名字放进 embedding
                # True 的话，collection name 也会参与生成向量

                inference_url="http://127.0.0.1:5000",
                # 指定 Transformer 推理 API 的地址
                # Weaviate 会把文本发送到这个本地服务
                # 由它生成 embedding 向量
            )
        ],

        # 定义每个对象要保存哪些字段
        properties=[
            Property(
                name="chunk",
                data_type=DataType.TEXT
            ),
            # chunk: 真正的文本块内容
            # 例如一个段落、一小段句子，或者固定长度切出来的文本

            Property(
                name="chapter_title",
                data_type=DataType.TEXT
            ),
            # chapter_title: 这个 chunk 来自哪一章

            Property(
                name="filename",
                data_type=DataType.TEXT
            ),
            # filename: 这个 chunk 来自哪个文件

            Property(
                name="chunking_strategy",
                data_type=DataType.TEXT,
                tokenization=Tokenization.FIELD
            ),
            # chunking_strategy: 使用了哪种 chunking 方法
            # 例如：
            # fixed_size_25
            # fixed_size_100
            # para_chunks
            # para_chunks_min_25
            #
            # tokenization=Tokenization.FIELD 表示：
            # 整个字段作为一个完整 token 处理
            # 适合做精确匹配 / filter
            #
            # 比如 fixed_size_25 会被整体看成一个标签，
            # 而不是拆成 fixed、size、25

            Property(
                name="chunk_index",
                data_type=DataType.INT
            ),
            # chunk_index: chunk 在原文中的顺序编号
            # 例如第 0 个 chunk、第 1 个 chunk、第 2 个 chunk
        ]
    )

else:
    # 如果 collection 已经存在，就直接获取它
    collection = client.collections.get("chunking_example")

<div class="alert alert-block alert-warning">  
<b>Note</b> <a class="tocSkip"></a><br>


```python
# Adding elements in the collection - this insertion should NOT run as the collection is already vectorized for you.
if len(collection) == 0:
    with collection.batch.fixed_size(batch_size=1, concurrent_requests=20) as batch:
        for chunking_strategy, chunk_objects in tqdm.tqdm(chunk_obj_sets.items()):
            for chunk_obj in chunk_objects:
                chunk_obj["chunking_strategy"] = chunking_strategy
                batch.add_object(
                    properties=chunk_obj,
                    uuid=generate_uuid5(chunk_obj)
                )
```

</div>

这段代码的作用是：统计 Weaviate collection 里一共有多少个 chunk 对象，并分别统计每种 chunking strategy 产生了多少个 chunk。

collection.aggregate.over_all().total_count

是 Weaviate Python client 提供的 API 方法 / 属性，不是 Python 自带的。


collection
  → 你的 Weaviate collection

.aggregate
  → 进入聚合统计功能

.over_all()
  → 对整个 collection 做统计

.total_count
  → 取出统计结果中的对象总数

 where_filter = Filter.by_property('chunking_strategy').equal(chunking_strategy)解释：


假设现在循环到：

chunking_strategy = "fixed_size_25"

那么这句代码：

where_filter = Filter.by_property('chunking_strategy').equal(chunking_strategy)

就等价于创建了一个条件：

只选择 chunking_strategy == "fixed_size_25" 的对象

也就是：

where chunking_strategy equals fixed_size_25

In [ ]:
# 打印整个 collection 中对象的总数量
# aggregate.over_all() 表示对整个 collection 做聚合统计
# total_count 表示对象总数
print(f"Total count: {collection.aggregate.over_all().total_count}")


# 遍历所有 chunking strategy
# chunk_obj_sets 是一个字典，大概长这样：
# {
#     "fixed_size_25": [...],
#     "fixed_size_100": [...],
#     "para_chunks": [...],
#     "para_chunks_min_25": [...]
# }
for chunking_strategy in chunk_obj_sets.keys():

    # 创建过滤条件：
    # 只选择 chunking_strategy 字段等于当前 strategy 的对象
    where_filter = Filter.by_property('chunking_strategy').equal(chunking_strategy)

    # 在 collection 中统计满足这个过滤条件的对象数量
    count = collection.aggregate.over_all(
        filters=where_filter
    ).total_count

    # 打印当前 chunking strategy 对应的对象数量
    print(f"Object count for {chunking_strategy}: {count}")

Total count: 1487
Object count for fixed_size_25: 672
Object count for fixed_size_100: 173
Object count for para_chunks: 549
Object count for para_chunks_min_25: 93


<a id='5'></a>
## 5 - Searching
---
In this section, you will explore semantic searching with different chunk sizes to visualize the impacts of the sizes in information retrieval.

.properties 可以理解成：这个 Weaviate object 里面保存的具体字段内容。

比如你创建 collection 时定义了这些字段：

Property(name="chunk", data_type=DataType.TEXT)
Property(name="chapter_title", data_type=DataType.TEXT)
Property(name="filename", data_type=DataType.TEXT)
Property(name="chunking_strategy", data_type=DataType.TEXT)
Property(name="chunk_index", data_type=DataType.INT)

那么每个检索结果 obj 里面的 .properties 大概就像一个 Python 字典：

obj.properties = {
    "chunk": "Git was created by Linus Torvalds in 2005.",
    "chapter_title": "01-introduction",
    "filename": "what-is-git.asc",
    "chunking_strategy": "fixed_size_100",
    "chunk_index": 3
}

所以：

obj.properties["chunk"]

取出来的是：

Git was created by Linus Torvalds in 2005.

而：

obj.properties["filename"]

取出来的是：

what-is-git.asc

In [ ]:
# 搜索文本
search_string = "history of git"


# 遍历所有 chunking strategy
# chunk_obj_sets 是一个字典，key 是不同的 chunking 方法
# 例如：
# fixed_size_25
# fixed_size_100
# para_chunks
# para_chunks_min_25
for chunking_strategy in chunk_obj_sets.keys():

    # 创建过滤条件
    # 只保留 chunking_strategy 字段等于当前 chunking_strategy 的对象
    #
    # 例如当前 chunking_strategy = "fixed_size_25"
    # 那么这个 filter 就表示：
    # 只搜索 chunking_strategy == "fixed_size_25" 的 chunks
    where_filter = Filter.by_property('chunking_strategy').equal(chunking_strategy)

    # 使用 near_text 做 semantic search / vector search
    # limit=2：
    # 每种 chunking strategy 只返回最相关的前 2 个结果
    response = collection.query.near_text(
        search_string,
        filters=where_filter,
        limit=2
    )

    # 打印当前是哪一种 chunking strategy 的检索结果
    print(f"RETRIEVED OBJECTS FOR CHUNKING STRATEGY {chunking_strategy.upper()}:\n")

    # 遍历搜索返回的对象
    # response.objects 是检索结果列表
    # enumerate 可以同时拿到编号 i 和对象 obj
    for i, obj in enumerate(response.objects):

        # 打印当前是第几个检索结果
        print(f"===== Object {i} =====")

        # obj.properties 是这个 Weaviate object 的字段内容
        # ['chunk'] 表示取出 chunk 这个字段，也就是真正的文本块内容
        print(f"{obj.properties['chunk']}")

        # 打印空行，让结果更清楚
        print()

RETRIEVED OBJECTS FOR CHUNKING STRATEGY FIXED_SIZE_25:

===== Object 0 =====
=== A Short History of Git As with many great things in life, Git began with a bit of creative destruction and fiery controversy. The

===== Object 1 =====
kernel efficiently (speed and data size) Since its birth in 2005, Git has evolved and matured to be easy to use and yet retain these initial qualities. It's amazingly fast,

RETRIEVED OBJECTS FOR CHUNKING STRATEGY FIXED_SIZE_100:

===== Object 0 =====
=== A Short History of Git As with many great things in life, Git began with a bit of creative destruction and fiery controversy. The Linux kernel is an open source software project of fairly large scope.(((Linux))) During the early years of the Linux kernel maintenance (1991–2002), changes to the software were passed around as patches and archived files. In 2002, the Linux kernel project began using a proprietary DVCS called BitKeeper.(((BitKeeper))) In 2005, the relationship between the community that develo

In this example, the query is a broad one focused on the "history of git." The results show that longer chunks tend to perform better. Upon examination, while the 25-word chunks might closely match the query in terms of semantic similarity, they lack sufficient context to significantly enhance the reader's understanding of the topic. Conversely, the paragraph chunks retrieved—particularly those with a minimum length of 25 words—provide comprehensive information that effectively educates the reader about the history of Git.

In [ ]:
search_string = "how to add the url of a remote repository"  # Or "available git remote commands"

for chunking_strategy in chunk_obj_sets.keys():
    where_filter = Filter.by_property('chunking_strategy').equal(chunking_strategy)
    response = collection.query.near_text(search_string, filters = where_filter, limit = 2)
    print(f"RETRIEVED OBJECTS FOR CHUNKING STRATEGY {chunking_strategy.upper()}:\n")
    for i, obj in enumerate(response.objects):
        print(f"===== Object {i} =====")
        print(f"{obj.properties['chunk']}")
        print()

RETRIEVED OBJECTS FOR CHUNKING STRATEGY FIXED_SIZE_25:

===== Object 0 =====
remote))) To add a new remote Git repository as a shortname you can reference easily, run `git remote add <shortname> <url>`: [source,console] ---- $ git remote origin $ git remote

===== Object 1 =====
manage your remote repositories. Remote repositories are versions of your project that are hosted on the Internet or network somewhere. You can have several of them, each of which generally

RETRIEVED OBJECTS FOR CHUNKING STRATEGY FIXED_SIZE_100:

===== Object 0 =====
adds the `origin` remote for you. Here's how to add a new remote explicitly.(((git commands, remote))) To add a new remote Git repository as a shortname you can reference easily, run `git remote add <shortname> <url>`: [source,console] ---- $ git remote origin $ git remote add pb https://github.com/paulboone/ticgit $ git remote -v origin https://github.com/schacon/ticgit (fetch) origin https://github.com/schacon/ticgit (push) pb https://github.com

In this example, the query was more specific, such as one made by a user looking to find out how to add the URL of a remote repository. Unlike the previous scenario, the 25-word chunks prove more useful here. Because the question was very specific, Weaviate could pinpoint the chunk with the most relevant passage—how to add a remote repository (`git remote add <shortname> <url>`).

Although other result sets contain some of this information, it's important to consider how the result will be used and displayed. Longer results might require more cognitive effort from the user to extract the relevant information.

这两个例子是在说明一个核心点：

chunk 不是越长越好，也不是越短越好，要看 query 是“宽泛问题”还是“具体问题”。

<a id='6'></a>
## 6 - Incorporating in a RAG system
---
Now you are familiar with chunking and you have a fully working collection, let's see how different chunk sizes impact text generation. Let's use a simple prompt.

In [ ]:
PROMPT = "Using this information and only this information, please explain {search_string} in a few short points.\nContext: {context}"

这段代码的作用是：为了公平比较不同 chunking strategy 的 RAG 效果，给短 chunk 多取一些，给长 chunk 少取一些，然后把检索到的 chunk 拼成 context，交给 LLM 生成回答。

比如你写了：

n_chunks_by_strat = dict()

n_chunks_by_strat['fixed_size_25'] = 8
n_chunks_by_strat['fixed_size_100'] = 2

这个字典就变成：

{
    "fixed_size_25": 8,
    "fixed_size_100": 2
}

In [ ]:
# 创建一个字典，用来规定：
# 每一种 chunking strategy 在检索时要取回多少个 chunks
n_chunks_by_strat = dict()


n_chunks_by_strat['fixed_size_25'] = 8

n_chunks_by_strat['para_chunks'] = 8

n_chunks_by_strat['fixed_size_100'] = 2

n_chunks_by_strat['para_chunks_min_25'] = 2


# 设置用户查询 query。
search_string = "history of git"  # Or "available git remote commands"

for chunking_strategy in chunk_obj_sets.keys():

    where_filter = Filter.by_property('chunking_strategy').equal(chunking_strategy)

    # limit=n_chunks_by_strat[chunking_strategy] 表示：
    # 根据当前 strategy 决定取回几个 chunks。
    # 例如：
    # 如果当前 chunking_strategy 是 fixed_size_25，
    # n_chunks_by_strat["fixed_size_25"] = 8，
    # 所以这里实际就是 limit=8。
    #
    # 如果当前 chunking_strategy 是 fixed_size_100，
    # n_chunks_by_strat["fixed_size_100"] = 2，
    # 所以这里实际就是 limit=2。
    response = collection.query.near_text(
        search_string,
        filters=where_filter,
        limit=n_chunks_by_strat[chunking_strategy]
    )

    # 创建一个空字符串，用来保存检索到的所有 chunk 内容。
    # 后面会把这些 chunk 拼接起来，作为 LLM 的 context。
    context_string = ""

    # 遍历 Weaviate 返回的检索结果。
    # response.objects 是检索出来的对象列表。
    for obj in response.objects:

        # obj.properties 是这个 object 的字段集合，
        # 里面包含 chunk、chapter_title、filename、chunking_strategy、chunk_index 等字段。
        #
        # obj.properties['chunk'] 表示取出真正的文本块内容。
        # 每取出一个 chunk，就加到 context_string 后面。
        # '\n' 是换行，方便不同 chunk 之间分开。
        context_string += obj.properties['chunk'] + '\n'

    # 把用户问题 search_string 和检索到的 context_string
    # 填入 PROMPT 模板，形成最终给 LLM 的 prompt。
    prompt = PROMPT.format(
        search_string=search_string,
        context=context_string
    )

    # 调用 LLM 生成回答。
    # role='assistant' 表示让模型以 assistant 的身份生成回复。
    response = generate_with_single_input(
        prompt,
        role='assistant'
    )

    # 打印当前使用的 query。
    print(f"Search string: {search_string}")

    # 打印当前使用的是哪一种 chunking strategy。
    # 这样可以比较不同 chunking 方法生成的回答效果。
    print(f"Chunking Strategy: {chunking_strategy}:")

    # 打印 LLM 最终生成的回答内容。
    print(f"Response:\n\t{response['content']}")

    # 打印空行，让不同 strategy 的结果更容易区分。
    print()

Search string: history of git
Chunking Strategy: fixed_size_25:
Response:
	Based solely on the provided text, here is the history of Git in short points:

*   **Origins and Evolution**: Historically associated with the Linux kernel for speed and efficiency, Git was born in **2005**. Since its birth, it has evolved to become user-friendly while retaining its core qualities of being amazingly fast and efficient with data sizes.
*   **Controversial Beginnings**: Like many great things, Git's beginning involved "creative destruction and fiery controversy."
*   **Development Timeline**: Specific development activity is noted from **2008** (e.g., March 15, 2008), where Scott Chacon made initial commits (including `Rakefile` and `simplegit.rb`) and performed cleanup tasks like removing unnecessary test diffs.

Search string: history of git
Chunking Strategy: fixed_size_100:
Response:
	Based on the text provided, here is the history of Git in a few short points:

*   **Linux Legacy (1991–2002)

In [ ]:
# Don't forget to close the client!
client.close()

Congratulations! You've finished the ungraded lab on Chunking! Keep it up!